<a href="https://colab.research.google.com/github/ljzier/ST-554-repo/blob/main/Zier_ST_554_HW9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Linda Zier**

**ST 554**

**HW #9**

**Goal**

• Finding a data set you can fit supervised learning models with

• Using a numeric or binary response, fitting three different classes of models and choosing an overall best model.

• Writing a narrative (via a notebook) with explanations and discussions as you go through the above.

**Data**



#Splitting the Data, Metrics, and Models

• Using spark MLlib, split the data into a training and test set.

• Choose and describe a metric you’ll be using to judge your models.

• You’ll be fitting three different classes of models. Briefly describe each model

#Model Fitting
Use Spark MLlib to fit your three different classes models to the training data. This
should be done using pipelines and cross validation to choose your best model for each model type. You
should compare your models using your metric chosen earlier.

• You should set up a pipeline in pyspark for each of your models

• You should do your transformations using the functions from MLlib to easily put them into the pipeline.

At least one of the pipelines should use four or more transformations prior to the model fit (estimator)

– VectorAssembler counts as a transformation

– Doing something like a log transform counts as well

– Adding polynomial terms or interaction terms counts

• You can use the same set of transformations for multiple models (if appropriate)

In [3]:
# clone my hub - just each first time I get started
!git clone https://github.com/ljzier/ST-554-repo.git

from pyspark.ml.feature import SQLTransformer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Setup CrossValidator() object and then use the .fit() method on crossval
lr = LinearRegression()
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.5]) \
    .addGrid(lr.elasticNetParam, [0, 0.2]) \
    .build()
crossval = CrossValidator(estimator = lr,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

# use pipeline to wrap transform, model, and predict into one easy call

pipeline = Pipeline(stages = [sqlTrans, assembler, lr])
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(),
                          numFolds=5)
cvModel = crossval.fit(train)
cvModel.transform(test) #for predictions

Cloning into 'ST-554-repo'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 244 (delta 97), reused 38 (delta 38), pack-reused 102 (from 1)
Receiving objects: 100% (244/244), 1.93 MiB | 6.44 MiB/s, done.
Resolving deltas: 100% (128/128), done.


In [8]:
df = pd.read_csv('ST-554-repo/data/SeoulBikeData.csv', encoding='latin1')
df.head()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


#Model Testing
Lastly, you should evaluate the best models from each class on the test set and state which overall model
was deemed the best.


In [ ]:
# assign new variable
red_wine['type'] = 0
white_wine['type'] = 1

#combine
wine = pd.concat([red_wine, white_wine])

#checking for null values. there is none
wine.isnull().sum()

wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


In [ ]:
#split data into training and text set including
# equal amounts of red and white wine

X_train, X_test, y_train, y_test = train_test_split(
  wine.drop("alcohol", axis = 1),
  wine["alcohol"],
  test_size=0.20,
  random_state=41, shuffle = True,
  stratify=wine['type'])

# I'm using means and stds like in class, but
means = X_train.apply(np.mean, axis = 0)
stds = X_train.apply(np.std, axis = 0)

X_train = X_train.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
2586,-0.324228,-0.966444,-0.127270,1.536375,0.161139,-0.644615,0.993523,1.114684,-1.362938,-0.075387,1.351979,0.571351
2000,-0.324228,-0.179618,-1.584610,-0.853833,-0.452065,1.443989,0.529082,-0.831770,-0.176317,-0.474290,-0.929266,0.571351
3032,-0.401439,-1.208544,0.983084,-0.811899,-0.563556,-0.870410,-0.417662,-0.686117,0.947850,-0.274839,-0.929266,0.571351
2410,-0.169806,-1.087494,-0.404859,0.383205,-0.256954,-0.023679,1.868815,0.498969,1.010303,0.323515,0.211357,0.571351
1195,0.525093,-0.724344,0.913687,1.829910,-0.507811,1.274643,0.457630,1.485437,-1.175577,-0.474290,0.211357,0.571351
...,...,...,...,...,...,...,...,...,...,...,...,...
4300,-0.633072,-0.300669,-0.127270,0.215471,4.258459,1.782681,0.725577,0.022286,-0.738401,-0.873192,-0.929266,0.571351
314,-1.096338,0.001957,-0.751845,0.236438,-0.452065,1.105296,0.922071,-0.202814,0.448220,0.589450,-0.929266,0.571351
1526,-0.324228,0.788782,-1.654007,-0.686099,0.216885,-0.701064,-1.382270,0.270559,0.510674,0.788901,0.211357,-1.750237
376,-0.633072,-0.179618,-0.266065,-0.832866,-0.535684,-1.039756,0.064642,-0.931079,0.635581,-0.141871,1.351979,0.571351
